In [1]:
# =====================================================================
# STEP 4.1 — Download PlantWild v1 + v2 (official HF) and inspect structure
# NEW notebook. Settings: GPU T4 x2, Internet ON.
# Raw zips go to a temp folder (NOT saved in output).
# =====================================================================
import os, zipfile, collections, time, json
from pathlib import Path
from huggingface_hub import hf_hub_download

RAW = Path("/kaggle/temp/plantwild_raw") if Path("/kaggle/temp").exists() else Path("/tmp/plantwild_raw")
RAW.mkdir(parents=True, exist_ok=True)
print("Raw folder:", RAW)

# ---- confirm PlantDoc input is attached ----
pd_manifest = list(Path("/kaggle/input").rglob("plantdoc_manifest.csv"))
print("PlantDoc manifest found at:", pd_manifest[0] if pd_manifest else "NOT FOUND (attach PlantDoc notebook output)")

# ---- download ----
zips = {}
for fname in ["plantwild.zip", "plantwild_v2.zip"]:
    t = time.time()
    p = hf_hub_download(repo_id="uqtwei2/PlantWild", filename=fname,
                        repo_type="dataset", local_dir=str(RAW))
    zips[fname] = Path(p)
    print(f"Downloaded {fname}: {Path(p).stat().st_size/1e9:.2f} GB in {time.time()-t:.0f}s")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif", ".gif", ".tif", ".tiff"}

def inspect_zip(zp, show_meta=True):
    print("\n" + "=" * 70)
    print("ZIP:", zp.name)
    print("=" * 70)
    with zipfile.ZipFile(zp) as z:
        names = [n for n in z.namelist() if not n.endswith("/")]
        print("Total files:", len(names))
        ext = collections.Counter(Path(n).suffix.lower() for n in names)
        print("Extensions:", dict(ext.most_common(12)))
        # folder tree (first 3 levels)
        lvl = [collections.Counter() for _ in range(3)]
        for n in names:
            parts = n.split("/")[:-1]
            for d in range(min(3, len(parts))):
                lvl[d]["/".join(parts[:d+1])] += 1
        for d in range(3):
            print(f"\nLevel-{d+1} folders: {len(lvl[d])}")
            for k, v in list(lvl[d].most_common())[:8]:
                print(f"   {k:70s} {v}")
        # non-image files (split lists, class names, descriptions)
        meta = [n for n in names if Path(n).suffix.lower() not in IMG_EXT]
        print(f"\nNon-image files: {len(meta)}")
        for n in meta[:40]:
            print("  ", n, f"({z.getinfo(n).file_size} bytes)")
        if show_meta:
            for n in meta[:12]:
                if Path(n).suffix.lower() in {".txt", ".json", ".csv", ".yaml", ".yml", ".md"}:
                    with z.open(n) as f:
                        head = f.read(700).decode("utf-8", errors="replace")
                    print(f"\n--- head of {n} ---\n{head}")
        return names

names_v1 = inspect_zip(zips["plantwild.zip"])
names_v2 = inspect_zip(zips["plantwild_v2.zip"], show_meta=True)

# ---- extract v1 only for now ----
EXT = RAW / "v1"
if not EXT.exists():
    t = time.time()
    with zipfile.ZipFile(zips["plantwild.zip"]) as z:
        z.extractall(EXT)
    print(f"\nExtracted v1 to {EXT} in {time.time()-t:.0f}s")
import shutil
print(f"Free disk in temp: {shutil.disk_usage(RAW).free/1e9:.1f} GB")
print("\nDONE — send ALL printed output.")

Raw folder: /tmp/plantwild_raw
PlantDoc manifest found at: /kaggle/input/datasets/mashfii/plantdoc-clean-thesis/plantdoc_manifest.csv


plantwild.zip:   0%|          | 0.00/2.71G [00:00<?, ?B/s]

Downloaded plantwild.zip: 2.71 GB in 19s


plantwild_v2.zip:   0%|          | 0.00/1.64G [00:00<?, ?B/s]

Downloaded plantwild_v2.zip: 1.64 GB in 15s

ZIP: plantwild.zip
Total files: 18547
Extensions: {'.jpg': 18540, '.txt': 3, '.png': 2, '.json': 2}

Level-1 folders: 1
   plantwild                                                              18547

Level-2 folders: 1
   plantwild/images                                                       18542

Level-3 folders: 89
   plantwild/images/basil leaf                                            589
   plantwild/images/citrus canker                                         535
   plantwild/images/cabbage leaf                                          464
   plantwild/images/apple leaf                                            444
   plantwild/images/squash leaf                                           410
   plantwild/images/grape downy mildew                                    395
   plantwild/images/cucumber leaf                                         348
   plantwild/images/tomato early blight                                   346

Non-image

In [2]:
# =====================================================================
# STEP 4.2 — PlantWild metadata: readme, official split decoding,
#            consistency checks, source-URL analysis, v1 <-> v2 mapping
# Same notebook/session as Step 4.1. CPU only, <1 min.
# =====================================================================
import json, zipfile, collections, re
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

RAW = Path("/tmp/plantwild_raw")
V1 = RAW / "v1" / "plantwild"
IMG_EXT = {".jpg", ".jpeg", ".png"}

# ---------------- 1) full readme ----------------
print("=" * 70); print("README (full)"); print("=" * 70)
print((V1 / "readme.txt").read_text(errors="replace"))

# ---------------- 2) classes + split ----------------
classes = {}
for line in (V1 / "classes.txt").read_text().strip().splitlines():
    k, name = line.split(" ", 1)
    classes[int(k)] = name.strip()
print("classes.txt entries:", len(classes), "| ids 0..", max(classes))

rows = []
for line in (V1 / "trainval.txt").read_text().strip().splitlines():
    parts = line.strip().split("=")
    if len(parts) != 3:
        print("BAD LINE:", line); continue
    rows.append({"rel": parts[0], "cid": int(parts[1]), "mode": int(parts[2])})
sp = pd.DataFrame(rows)
sp["folder"] = sp.rel.str.split("/").str[0]
print("\nSplit lines:", len(sp), "| unique paths:", sp.rel.nunique())

print("\n=== MODE COUNTS ===")
mc = sp["mode"].value_counts().sort_index()
print(pd.DataFrame({"n": mc, "pct": (100 * mc / mc.sum()).round(2)}).to_string())

# consistency
files = sorted(str(p.relative_to(V1 / "images")) for p in (V1 / "images").rglob("*")
               if p.suffix.lower() in IMG_EXT)
fs, ss = set(files), set(sp.rel)
print("\nImages on disk:", len(fs))
print("In split but missing on disk:", len(ss - fs), list(ss - fs)[:5])
print("On disk but not in split   :", len(fs - ss), list(fs - ss)[:5])
mism = sp[sp.folder != sp.cid.map(classes)]
print("Rows where folder name != classes.txt name for cid:", len(mism))
if len(mism): print(mism.head(10).to_string())

print("\n=== PER-CLASS SPLIT (sorted by total) ===")
pc = sp.pivot_table(index="folder", columns="mode", values="rel", aggfunc="count").fillna(0).astype(int)
pc["total"] = pc.sum(1)
pc = pc.sort_values("total")
print("Smallest 10 classes:"); print(pc.head(10).to_string())
print("Largest 5 classes:");   print(pc.tail(5).to_string())
for m in sorted(sp["mode"].unique()):
    print(f"mode {m}: min per class = {pc[m].min()}, classes with <5 = {(pc[m] < 5).sum()}")

# ---------------- 3) prompts ----------------
prompts = json.loads((V1 / "plantwild_prompts.json").read_text())
npr = pd.Series({k: len(v) for k, v in prompts.items()})
print(f"\nPrompts: {len(prompts)} classes | per class min/median/max = "
      f"{npr.min()}/{int(npr.median())}/{npr.max()}")
print("Classes without prompts:", sorted(set(classes.values()) - set(prompts))[:10])

# ---------------- 4) source URLs ----------------
urls = json.loads((V1 / "url_record.json").read_text())
u = pd.DataFrame({"key": list(urls.keys()), "url": list(urls.values())})
u["rel"] = u.key.str.replace("plantwild/images/", "", regex=False)
u = u.merge(sp[["rel", "mode", "folder"]], on="rel", how="right")
print("\n=== SOURCE URLs ===")
print("Images with URL record:", int(u.url.notna().sum()), "/", len(u))
u["domain"] = u.url.fillna("").map(lambda s: urlparse(s).netloc.replace("www.", "") if s else "NONE")
print("Unique URLs:", u.url.nunique(), "| unique domains:", u.domain.nunique())
print("Top 12 domains:"); print(u.domain.value_counts().head(12).to_string())
g = u.dropna(subset=["url"]).groupby("url")
multi = g.size()[g.size() > 1]
print(f"\nURLs shared by >1 image: {len(multi)} | images in them: {int(multi.sum())} "
      f"| max images from one URL: {int(multi.max()) if len(multi) else 0}")
cross = g["mode"].nunique()
print("URL groups spanning >1 split mode:", int((cross > 1).sum()),
      "| images in them:", int(g.size()[cross > 1].sum()))
cls_cross = g["folder"].nunique()
print("URL groups spanning >1 CLASS:", int((cls_cross > 1).sum()))
print("Example cross-split URL group:")
ex = cross[cross > 1].index[:2]
print(u[u.url.isin(ex)][["rel", "mode", "url"]].to_string(index=False))

# ---------------- 5) v1 <-> v2 ----------------
with zipfile.ZipFile(RAW / "plantwild_v2.zip") as z:
    v2 = [n for n in z.namelist() if Path(n).suffix.lower() in IMG_EXT]
v2df = pd.DataFrame({"n": v2})
v2df["folder"] = v2df.n.str.split("/").str[1]
v2df["fname"] = v2df.n.str.split("/").str[-1]
v1c, v2c = set(sp.folder), set(v2df.folder)
print("\n=== v1 vs v2 CLASSES ===")
print(f"v1: {len(v1c)} | v2: {len(v2c)} | shared names: {len(v1c & v2c)}")
print("In v1 only:", sorted(v1c - v2c))
print("In v2 only:", sorted(v2c - v1c))

sp["fname"] = sp.rel.str.split("/").str[-1]
key1 = set(zip(sp.folder, sp.fname)); key2 = set(zip(v2df.folder, v2df.fname))
print(f"\nv1 images whose (class, filename) exists in v2: {len(key1 & key2)} / {len(key1)}")
shared = sorted(v1c & v2c)
t = pd.DataFrame({
    "v1": sp[sp.folder.isin(shared)].groupby("folder").size(),
    "v2": v2df[v2df.folder.isin(shared)].groupby("folder").size(),
    "v1_kept_by_name": pd.Series(collections.Counter(f for f, n in key1 & key2)),
}).fillna(0).astype(int)
t["kept_%"] = (100 * t.v1_kept_by_name / t.v1).round(1)
print("Shared classes, sorted by kept %: (first 12)")
print(t.sort_values("kept_%").head(12).to_string())
print(f"Overall kept by name in shared classes: {t.v1_kept_by_name.sum()} / {t.v1.sum()}")
print("v2 filename examples:", v2df.fname.sample(8, random_state=0).tolist())
print("v1 filename examples:", sp.fname.sample(8, random_state=0).tolist())

sp.to_csv("/kaggle/working/plantwild_v1_split_raw.csv", index=False)
print("\nSaved /kaggle/working/plantwild_v1_split_raw.csv. DONE — send ALL output.")

README (full)
PlantWild Dataset


IMAGES AND CLASS LABELS:
Images are contained in the directory images/, with 89 subdirectories.


------- List of class names (classes.txt) ------
The list of class names is contained in the file classes.txt, with each line corresponding to one class:

<class_id> <class_name>



------- Train/val/test split (split.txt) ------
The suggested train/val/test split is contained in the file split.txt, with each line corresponding to one image:

<image_path>=<class_id>=<mode>

where <image_path> corresponds to the relative paths of images, <class_id> correspond the IDs in classes.txt, and a value of 0, 1, 2 for <mode> denotes that the file is in the test, training or validation set, respectively.



------- Text prompts for each class (plantwild_promptps.json) ------
The GPT-generated descriptive prompts, taken as the input of CLIP's textual encoder. For each class there are 50 prompts.


------- ULRs of image sources (url_record.json) ------
The record file 

/tmp/ipykernel_58/2403849399.py:110: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  }).fillna(0).astype(int)


In [3]:
# =====================================================================
# STEP 4.3 — PlantWild v1: RGB conversion, pre-resize, integrity,
#            master manifest (official roles, URLs, "Copy" files)
# Same session as 4.1/4.2. CPU, ~4-7 min.
# =====================================================================
import json, re, collections, time
from pathlib import Path
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image, ImageOps

RAW  = Path("/tmp/plantwild_raw/v1/plantwild")
DST  = Path("/kaggle/working/proc/plantwild_v1")
W    = Path("/kaggle/working")
SHORT = 320
ROLE = {0: "test", 1: "train", 2: "val"}          # from official readme

sp = pd.read_csv(W / "plantwild_v1_split_raw.csv")
urls = json.loads((RAW / "url_record.json").read_text())
classes = {}
for line in (RAW / "classes.txt").read_text().strip().splitlines():
    k, n = line.split(" ", 1); classes[int(k)] = n.strip()

sp = sp.sort_values("rel").reset_index(drop=True)
sp["uid"] = [f"pw1_{i:05d}" for i in range(len(sp))]
sp["class_name"] = sp.cid.map(classes)
sp["class_id"] = sp.cid
sp["official_role"] = sp["mode"].map(ROLE)
sp["url"] = sp.rel.map(lambda r: urls.get(f"plantwild/images/{r}", ""))
sp["domain"] = sp.url.map(lambda s: urlparse(s).netloc.replace("www.", "") if s else "")
copy_re = re.compile(r"\s*-\s*copy(\s*\(\d+\))?", re.I)
sp["copy_flag"] = sp.fname.str.contains(r"copy", case=False).astype(int)
sp["copy_base"] = sp.fname.map(lambda f: copy_re.sub("", Path(f).stem) + Path(f).suffix)
sp["src_prefix"] = sp.fname.map(lambda f: (re.match(r"([A-Za-z]+)_", f).group(1).lower()
                                          if re.match(r"([A-Za-z]+)_", f) else
                                          ("numeric" if re.match(r"\d", f) else "other")))

def process(i):
    r = sp.iloc[i]
    src = RAW / "images" / r.rel
    rec = {"uid": r.uid}
    try:
        with Image.open(src) as im:
            rec["orig_mode"] = im.mode
            im = ImageOps.exif_transpose(im)
            rec["orig_w"], rec["orig_h"] = im.size
            if im.mode in ("RGBA", "LA", "P"):
                im = im.convert("RGBA")
                bg = Image.new("RGB", im.size, (255, 255, 255))
                bg.paste(im, mask=im.split()[-1]); im = bg
            else:
                im = im.convert("RGB")
            w, h = im.size; s = min(w, h)
            if s > SHORT:
                k = SHORT / s
                im = im.resize((max(1, round(w * k)), max(1, round(h * k))), Image.BICUBIC)
            out = DST / r.official_role / r.class_name / f"{r.uid}.jpg"
            out.parent.mkdir(parents=True, exist_ok=True)
            im.save(out, "JPEG", quality=95)
            rec.update(proc_path=str(out), proc_w=im.size[0], proc_h=im.size[1],
                       small_flag=int(s < 224), ok=1)
    except Exception as e:
        rec.update(ok=0, error=f"{type(e).__name__}: {e}")
    return rec

t = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    recs = list(ex.map(process, range(len(sp))))
df = sp.merge(pd.DataFrame(recs), on="uid")
df["dataset"] = "plantwild_v1"
print(f"Processed in {time.time()-t:.0f}s")

# ---------------- report ----------------
print("\n=== PROCESSING ===")
print("OK:", int(df.ok.sum()), "| failed:", int((df.ok == 0).sum()))
if (df.ok == 0).any():
    print(df[df.ok == 0][["rel", "error"]].head(20).to_string())
print("Original modes:", dict(collections.Counter(df.orig_mode.dropna())))
print("Roles:", df.official_role.value_counts().to_dict())

ok = df[df.ok == 1]
w, h = ok.orig_w, ok.orig_h
print(f"\nWidth  min/median/max: {int(w.min())} / {int(w.median())} / {int(w.max())}")
print(f"Height min/median/max: {int(h.min())} / {int(h.median())} / {int(h.max())}")
print("Small (<224 short side):", int(ok.small_flag.sum()),
      f"({100*ok.small_flag.mean():.1f}%) | by role:",
      ok.groupby("official_role").small_flag.sum().to_dict())
sm = ok[ok.small_flag == 1].class_name.value_counts()
print("Classes with most small images:"); print(sm.head(8).to_string())
healthy = ok.class_name.str.endswith(" leaf")
print(f"Small-image rate: healthy classes {100*ok[healthy].small_flag.mean():.1f}% | "
      f"disease classes {100*ok[~healthy].small_flag.mean():.1f}%")
ar = w / h
print(f"Aspect ratio extreme (>2.5 or <0.4): {int(((ar > 2.5) | (ar < 0.4)).sum())}")
df["ar_flag"] = ((df.orig_w / df.orig_h > 2.5) | (df.orig_w / df.orig_h < 0.4)).astype(int)

print("\n=== SOURCE PREFIX ===")
print(pd.crosstab(df.src_prefix, df.official_role).to_string())

print("\n=== 'COPY' FILES ===")
cp = df[df.copy_flag == 1]
print("Copy-named files:", len(cp), "| classes affected:", cp.class_name.nunique())
print("By role:", cp.official_role.value_counts().to_dict())
print("Top classes:"); print(cp.class_name.value_counts().head(8).to_string())
base = df[df.copy_flag == 0].set_index(["class_name", "fname"]).official_role
has_orig = [(c, b) in base.index for c, b in zip(cp.class_name, cp.copy_base)]
cp = cp.assign(orig_exists=has_orig)
cp["orig_role"] = [base.get((c, b), None) for c, b in zip(cp.class_name, cp.copy_base)]
print("Copies whose original exists in same class:", int(cp.orig_exists.sum()), "/", len(cp))
cross = cp[cp.orig_exists & (cp.orig_role != cp.official_role)]
print("Copy and original in DIFFERENT split roles:", len(cross))
print(pd.crosstab(cross.orig_role, cross.official_role).to_string() if len(cross) else "")
print("Examples:"); print(cp[["rel", "official_role", "orig_role"]].head(6).to_string(index=False))

tot = sum(f.stat().st_size for f in DST.rglob("*.jpg"))
print(f"\nProcessed folder size: {tot/1e9:.2f} GB")
df.to_csv(W / "plantwild_v1_manifest.csv", index=False)
with open(W / "plantwild_v1_classes.json", "w") as f:
    json.dump([classes[k] for k in sorted(classes)], f, indent=1)
print("Saved plantwild_v1_manifest.csv + classes json. DONE — send ALL output.")

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


Processed in 137s

=== PROCESSING ===
OK: 18542 | failed: 0
Original modes: {'RGB': 18520, 'P': 1, 'RGBA': 19, 'L': 1, 'CMYK': 1}
Roles: {'train': 13045, 'test': 3677, 'val': 1820}

Width  min/median/max: 36 / 645 / 6943
Height min/median/max: 39 / 513 / 5408
Small (<224 short side): 1355 (7.3%) | by role: {'test': 262, 'train': 949, 'val': 144}
Classes with most small images:
class_name
potato early blight          63
potato late blight           61
rice sheath blight           55
banana panama disease        54
tomato early blight          43
tomato leaf mold             38
tomato septoria leaf spot    38
grape black rot              32
Small-image rate: healthy classes 2.3% | disease classes 10.9%
Aspect ratio extreme (>2.5 or <0.4): 161

=== SOURCE PREFIX ===
official_role  test  train  val
src_prefix                     
add              22     50    6
google         1874   6649  908
numeric        1716   6135  873
other            36    133   18
zgoogle          29     78   15

=

In [4]:
# =====================================================================
# STEP 4.4 — PlantWild v1 duplicate detection (pHash + DINOv2 ViT-B/14)
# Same session. GPU T4. ~5-8 min.
# =====================================================================
import numpy as np, pandas as pd, torch, timm, imagehash, time
from PIL import Image
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

W = Path("/kaggle/working")
df = pd.read_csv(W / "plantwild_v1_manifest.csv")
df = df[df.ok == 1].reset_index(drop=True)
N = len(df); print("Images:", N)
role = df.official_role.values; cls = df.class_name.values

# ---------------- 1) pHash (packed uint64) ----------------
t = time.time()
def ph(p):
    with Image.open(p) as im:
        return int(str(imagehash.phash(im.convert("RGB"), hash_size=8)), 16)
with ThreadPoolExecutor(max_workers=8) as ex:
    H = np.array(list(ex.map(ph, df.proc_path)), dtype=np.uint64)
print(f"pHash done in {time.time()-t:.0f}s")

# ---------------- 2) DINOv2 embeddings ----------------
t = time.time()
dev = "cuda"
model = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True,
                          num_classes=0, img_size=224).to(dev).eval().half()
cfg = timm.data.resolve_data_config({}, model=model)
tf = timm.data.create_transform(**{**cfg, "input_size": (3, 224, 224), "crop_pct": 1.0})
class DS(torch.utils.data.Dataset):
    def __len__(self): return N
    def __getitem__(self, i):
        with Image.open(df.proc_path[i]) as im:
            return tf(im.convert("RGB"))
dl = torch.utils.data.DataLoader(DS(), batch_size=256, num_workers=4, pin_memory=True)
feats = []
with torch.no_grad():
    for x in dl:
        feats.append(model(x.to(dev, non_blocking=True).half()).float())
E = torch.cat(feats); E = torch.nn.functional.normalize(E, dim=1)
np.save(W / "plantwild_v1_dinov2b_224.npy", E.cpu().numpy())
df[["uid"]].to_csv(W / "plantwild_v1_dinov2b_224_uids.csv", index=False)
print(f"Embeddings {tuple(E.shape)} in {time.time()-t:.0f}s")

# ---------------- 3) candidate pairs ----------------
t = time.time()
K = 30
pairs = set(); nn1 = np.zeros(N, dtype=np.float32)
Eh = E.half()
for s in range(0, N, 1024):
    S = (Eh[s:s+1024] @ Eh.T).float()
    S[torch.arange(S.shape[0]), torch.arange(s, s + S.shape[0])] = -1
    v, ix = S.topk(K, dim=1)
    nn1[s:s+S.shape[0]] = v[:, 0].cpu().numpy()
    v, ix = v.cpu().numpy(), ix.cpu().numpy()
    for r in range(v.shape[0]):
        i = s + r
        for val, j in zip(v[r], ix[r]):
            if val < 0.90: break
            pairs.add((min(i, j), max(i, j)))
n_cos = len(pairs)
for s in range(0, N, 512):                         # pHash Hamming via popcount
    d = np.bitwise_count(H[s:s+512, None] ^ H[None, :])
    ii, jj = np.nonzero(d <= 12)
    for a, b in zip(ii + s, jj):
        if a < b: pairs.add((int(a), int(b)))
print(f"Candidate pairs: {len(pairs)} (cos>=0.90 top-{K}: {n_cos}) in {time.time()-t:.0f}s")

P = pd.DataFrame(list(pairs), columns=["i", "j"])
Ec = E.cpu().numpy()
P["cos"] = np.einsum("ij,ij->i", Ec[P.i], Ec[P.j])
P["ham"] = np.bitwise_count(H[P.i] ^ H[P.j]).astype(int)
ri, rj = role[P.i], role[P.j]
P["type"] = ["-".join(sorted([a, b])) for a, b in zip(ri, rj)]
P["same_class"] = cls[P.i] == cls[P.j]
P["same_url"] = (df.url.values[P.i] == df.url.values[P.j]) & (df.url.values[P.i] != "")
P["uid_i"] = df.uid.values[P.i]; P["uid_j"] = df.uid.values[P.j]
P["class_i"] = cls[P.i]; P["class_j"] = cls[P.j]

# ---------------- 4) validation with "Copy" files ----------------
key = {(c, f): k for k, (c, f) in enumerate(zip(df.class_name, df.fname))}
cp = df[(df.copy_flag == 1)]
cp_pairs = [(k, key[(c, b)]) for k, c, b in zip(cp.index, cp.class_name, cp.copy_base)
            if (c, b) in key]
cpi = np.array(cp_pairs)
cc = np.einsum("ij,ij->i", Ec[cpi[:, 0]], Ec[cpi[:, 1]])
ch = np.bitwise_count(H[cpi[:, 0]] ^ H[cpi[:, 1]])
print("\n=== VALIDATION: Copy file vs its original ===")
print(f"pairs: {len(cpi)} | pHash<=4: {(ch <= 4).mean():.3f} | pHash<=8: {(ch <= 8).mean():.3f}"
      f" | cos>=0.98: {(cc >= 0.98).mean():.3f} | cos min/median: {cc.min():.3f}/{np.median(cc):.3f}")
print("Copy pairs NOT caught by (pHash<=8 or cos>=0.98):", int(((ch > 8) & (cc < 0.98)).sum()))

# ---------------- 5) statistics ----------------
print("\n=== PAIR COUNTS BY RULE ===")
rules = [("pHash<=4", P.ham <= 4), ("pHash<=8", P.ham <= 8), ("cos>=0.95", P.cos >= 0.95),
         ("cos>=0.98", P.cos >= 0.98), ("STRICT: pHash<=8|cos>=0.98", (P.ham <= 8) | (P.cos >= 0.98))]
rows = []
for name, m in rules:
    s = P[m]
    r = {"rule": name, "total": len(s), "diff_class": int((~s.same_class).sum()),
         "same_url": int(s.same_url.sum())}
    r.update(s.type.value_counts().to_dict()); rows.append(r)
print(pd.DataFrame(rows).fillna(0).set_index("rule").astype(int).to_string())

strict = P[(P.ham <= 8) | (P.cos >= 0.98)]
print("\n=== STRICT duplicates: test images with a twin in train or val ===")
tt = strict[strict.type.isin(["test-train", "test-val"])]
test_ids = set(np.where(role[tt.i] == "test", tt.i, tt.j))
print(f"test images affected: {len(test_ids)} / {int((role == 'test').sum())} "
      f"({100*len(test_ids)/(role == 'test').sum():.2f}%)")
print("  of which are Copy-named or have a Copy-named twin:",
      int(sum(1 for _, r in tt.iterrows()
              if df.copy_flag[r.i] or df.copy_flag[r.j])), "pairs")

print("\n=== STRICT duplicates with DIFFERENT labels ===")
dc = strict[~strict.same_class]
print("pairs:", len(dc), "| cross-split:", int((dc.type.str.contains("-") &
      ~dc.type.isin(["train-train", "test-test", "val-val"])).sum()))
agg = dc.apply(lambda r: " <-> ".join(sorted([r.class_i, r.class_j])), axis=1).value_counts()
print(agg.head(15).to_string())

print("\n=== URL signal ===")
su = P[P.same_url]
print(f"candidate pairs sharing a URL: {len(su)} | of these STRICT duplicates: "
      f"{int(((su.ham <= 8) | (su.cos >= 0.98)).sum())}")
print(f"STRICT duplicates sharing a URL: {int(strict.same_url.sum())} / {len(strict)}")

print("\n=== nearest-neighbour cosine distribution ===")
print(pd.Series(nn1).describe(percentiles=[.5, .9, .95, .99]).round(3).to_string())
band = P[(P.cos >= 0.93) & (P.cos < 0.98) & (P.ham > 8)]
print("Borderline pairs for ORB (0.93<=cos<0.98, ham>8):", len(band),
      band.type.value_counts().to_dict())

P.to_csv(W / "plantwild_v1_dup_candidates.csv", index=False)
np.save(W / "plantwild_v1_phash.npy", H)
print("\nSaved candidates, embeddings, pHash. DONE — send ALL output.")

Images: 18542
pHash done in 34s


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Embeddings (18542, 768) in 66s
Candidate pairs: 10512 (cos>=0.90 top-30: 9675) in 2s

=== VALIDATION: Copy file vs its original ===
pairs: 584 | pHash<=4: 0.000 | pHash<=8: 0.000 | cos>=0.98: 0.000 | cos min/median: 0.009/0.575
Copy pairs NOT caught by (pHash<=8 or cos>=0.98): 584

=== PAIR COUNTS BY RULE ===
                            total  diff_class  same_url  train-train  test-train  train-val  test-val  test-test  val-val
rule                                                                                                                     
pHash<=4                      823         525       444          380         223        131        42         37       10
pHash<=8                     1013         582       473          473         271        159        53         42       15
cos>=0.95                    1482         604       550          712         396        235        64         56       19
cos>=0.98                     728         484       434          348         20

In [5]:
# =====================================================================
# STEP 4.5 — PlantWild v1: ORB+RANSAC verification, final duplicate
#            groups, PlantWild-Clean flags (P1 official / P2 clean)
# Same session. CPU, ~3-6 min.
# =====================================================================
import numpy as np, pandas as pd, cv2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

W = Path("/kaggle/working")
df = pd.read_csv(W / "plantwild_v1_manifest.csv")
df = df[df.ok == 1].reset_index(drop=True)
P = pd.read_csv(W / "plantwild_v1_dup_candidates.csv")
E = np.load(W / "plantwild_v1_dinov2b_224.npy")
paths = df.proc_path.values; role = df.official_role.values; cls = df.class_name.values
rng = np.random.default_rng(42)

# ---------------- ORB + RANSAC ----------------
def feat(i, flip=False):
    g = cv2.imread(paths[i], cv2.IMREAD_GRAYSCALE)
    if flip: g = cv2.flip(g, 1)
    return cv2.ORB_create(nfeatures=1500).detectAndCompute(g, None)

def inl(a, b, flip):
    (k1, d1), (k2, d2) = feat(a), feat(b, flip)
    if d1 is None or d2 is None or len(k1) < 8 or len(k2) < 8: return 0
    m = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(d1, d2, k=2)
    good = [x for x, *y in m if y and x.distance < 0.75 * y[0].distance]
    if len(good) < 8: return len(good) // 2
    src = np.float32([k1[g.queryIdx].pt for g in good]).reshape(-1, 1, 2)
    dst = np.float32([k2[g.trainIdx].pt for g in good]).reshape(-1, 1, 2)
    _, mk = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    return 0 if mk is None else int(mk.sum())

def verify(a, b): return max(inl(a, b, False), inl(a, b, True))
def run(pairs):
    with ThreadPoolExecutor(max_workers=4) as ex:
        return np.array(list(ex.map(lambda t: verify(*t), pairs)))

# ---------------- calibration ----------------
pos = P[P.ham <= 4][["i", "j"]].sample(min(300, int((P.ham <= 4).sum())), random_state=0).values.tolist()
neg = []
while len(neg) < 300:
    a, b = rng.integers(0, len(df), 2)
    if a != b and cls[a] == cls[b] and 0.80 <= float(E[a] @ E[b]) < 0.90:
        neg.append([int(a), int(b)])
pos_in, neg_in = run(pos), run(neg)
print("=== CALIBRATION ===")
print(f"positives (pHash<=4): n={len(pos_in)} min={pos_in.min()} p5={np.percentile(pos_in,5):.0f} median={np.median(pos_in):.0f}")
print(f"negatives (cos .80-.90 same class): n={len(neg_in)} median={np.median(neg_in):.0f} "
      f"p95={np.percentile(neg_in,95):.0f} p99={np.percentile(neg_in,99):.0f} max={neg_in.max()}")
THR = int(max(15, np.ceil(np.percentile(neg_in, 99) * 1.5)))
print(f"threshold={THR} | recall on positives={(pos_in >= THR).mean():.3f} | FPR on negatives={(neg_in >= THR).mean():.3f}")

# ---------------- verify cross-label strict duplicates ----------------
strict = P[(P.ham <= 8) | (P.cos >= 0.98)]
dc = strict[~strict.same_class]
dc_in = run(dc[["i", "j"]].values.tolist())
print("\n=== CROSS-LABEL STRICT DUPLICATES: ORB check ===")
print(f"pairs: {len(dc)} | ORB-confirmed (>=THR): {(dc_in >= THR).mean():.3f} | "
      f"median inliers: {np.median(dc_in):.0f}")

# ---------------- verify borderline band ----------------
band = P[(P.cos >= 0.93) & (P.cos < 0.98) & (P.ham > 8)].copy()
band["inliers"] = run(band[["i", "j"]].values.tolist())
band["verified"] = band.inliers >= THR
print("\n=== BORDERLINE PAIRS ===")
print("total:", len(band), "| verified:", int(band.verified.sum()))
print(band.groupby("type").verified.agg(["size", "sum"]).to_string())
print(band.groupby(pd.cut(band.cos, [0.93, 0.95, 0.96, 0.97, 0.98]), observed=False)
      .verified.agg(["size", "mean"]).round(2).to_string())

# ---------------- final groups ----------------
edges = pd.concat([strict[["i", "j"]], band[band.verified][["i", "j"]]]).drop_duplicates()
N = len(df); parent = list(range(N))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
for a, b in edges.values:
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb
df["dup_group"] = [find(k) for k in range(N)]

# ---------------- cleaning rules ----------------
prio = {"test": 0, "val": 1, "train": 2}
df["drop_reason"] = ""
for g, sub in df.groupby("dup_group"):
    if len(sub) == 1: continue
    if sub.class_name.nunique() > 1:
        df.loc[sub.index, "drop_reason"] = "label_conflict"; continue
    keep = sub.assign(p=sub.official_role.map(prio)).sort_values(["p", "uid"]).index[0]
    for k in sub.index:
        if k == keep: continue
        df.loc[k, "drop_reason"] = ("leak_copy" if df.official_role[k] != df.official_role[keep]
                                    else f"dup_within_{df.official_role[k]}")
df["in_clean"] = (df.drop_reason == "").astype(int)
df["p1_role"] = df.official_role
df["p2_role"] = np.where(df.in_clean == 1, df.official_role, "excluded")

gs = df.groupby("dup_group").size()
print("\n=== FINAL GROUPS ===")
print("groups >1:", int((gs > 1).sum()), "| images in them:", int(gs[gs > 1].sum()),
      "| largest:", int(gs.max()))
mixed = df.groupby("dup_group").official_role.nunique()
print("groups spanning >1 official split:", int((mixed > 1).sum()))
t_aff = df[(df.official_role == "test") &
           df.dup_group.isin(mixed[mixed > 1].index)]
print(f"official TEST images with a twin in train/val (final): {len(t_aff)} "
      f"({100*len(t_aff)/(df.official_role=='test').sum():.2f}%)")

print("\n=== DROP REASONS ===")
print(pd.crosstab(df.drop_reason.replace("", "KEPT"), df.official_role).to_string())

print("\n=== OFFICIAL vs CLEAN per role ===")
for r in ["train", "val", "test"]:
    o = (df.official_role == r).sum(); c = ((df.official_role == r) & (df.in_clean == 1)).sum()
    print(f"{r:5s}: {o} -> {c}  (-{o-c}, {100*(o-c)/o:.1f}%)")

pc = df[df.in_clean == 1].pivot_table(index="class_name", columns="official_role",
                                      values="uid", aggfunc="count").fillna(0).astype(int)
print("\nClean per-class minimum:", pc.min().to_dict())
print("Classes with <5 clean test images:", pc[pc.test < 5].index.tolist())
print("Classes with <3 clean val images:", pc[pc.val < 3].index.tolist())
lost = (1 - pc.sum(1) / df.groupby("class_name").size()).sort_values(ascending=False)
print("Classes losing the most images (%):")
print((100 * lost.head(10)).round(1).to_string())

df.to_csv(W / "plantwild_v1_manifest.csv", index=False)
edges.to_csv(W / "plantwild_v1_dup_edges_final.csv", index=False)
print("\nManifest updated (dup_group, drop_reason, in_clean, p1_role, p2_role). DONE — send output.")

=== CALIBRATION ===
positives (pHash<=4): n=300 min=0 p5=139 median=1196
negatives (cos .80-.90 same class): n=300 median=4 p95=7 p99=10 max=102
threshold=16 | recall on positives=0.983 | FPR on negatives=0.003

=== CROSS-LABEL STRICT DUPLICATES: ORB check ===
pairs: 594 | ORB-confirmed (>=THR): 0.958 | median inliers: 1245

=== BORDERLINE PAIRS ===
total: 1823 | verified: 834
             size  sum
type                  
test-test      66   24
test-train    474  224
test-val       70   32
train-train   871  414
train-val     307  127
val-val        35   13
              size  mean
cos                     
(0.93, 0.95]  1241  0.31
(0.95, 0.96]   310  0.68
(0.96, 0.97]   181  0.86
(0.97, 0.98]    91  0.95

=== FINAL GROUPS ===
groups >1: 1126 | images in them: 2596 | largest: 12
groups spanning >1 official split: 551
official TEST images with a twin in train/val (final): 414 (11.26%)

=== DROP REASONS ===
official_role     test  train   val
drop_reason                        
KEPT      

In [6]:
# =====================================================================
# STEP 4.6 — Sensitivity of the PlantWild-Clean result to the ORB threshold
# Run in the SAME session right after Step 4.5 (uses df, strict, band).
# CPU, <1 min.
# =====================================================================
import numpy as np, pandas as pd
from pathlib import Path
W = Path("/kaggle/working")

band.to_csv(W / "plantwild_v1_borderline_orb.csv", index=False)   # keep inliers for later

base = df[["uid", "class_name", "official_role"]].copy()
N = len(base)
prio = {"test": 0, "val": 1, "train": 2}

def evaluate(edges):
    parent = list(range(N))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    g = np.array([find(k) for k in range(N)])
    t = base.assign(g=g)
    n_cls = t.groupby("g").class_name.nunique()
    n_role = t.groupby("g").official_role.nunique()
    size = t.groupby("g").size()
    conflict = t.g.isin(n_cls[n_cls > 1].index)
    test_leak = ((t.official_role == "test") & t.g.isin(n_role[n_role > 1].index)).sum()
    # clean test size: test images not in conflict groups, and first test image kept per group
    ok_t = t[(t.official_role == "test") & ~conflict]
    clean_test = ok_t.g.nunique() + 0  # one kept per group (test has top priority)
    return {"dup_groups>1": int((size > 1).sum()),
            "test_leak": int(test_leak),
            "test_leak_%": round(100 * test_leak / (base.official_role == "test").sum(), 2),
            "conflict_imgs": int(conflict.sum()),
            "clean_test": int(clean_test)}

rows = []
strict_e = list(map(tuple, strict[["i", "j"]].values))
rows.append({"setting": "strict only (no ORB)", **evaluate(strict_e)})
for thr in [16, 50, 100, 150, 286]:
    e = strict_e + list(map(tuple, band[band.inliers >= thr][["i", "j"]].values))
    rows.append({"setting": f"strict + ORB>={thr}", **evaluate(e)})
res = pd.DataFrame(rows).set_index("setting")
print("=== SENSITIVITY: PlantWild v1 ===")
print(res.to_string())
res.to_csv(W / "plantwild_v1_threshold_sensitivity.csv")

print("\nInlier distribution of borderline pairs:")
print(pd.cut(band.inliers, [-1, 10, 16, 50, 100, 150, 286, 10**6]).value_counts().sort_index().to_string())
print("\nSaved plantwild_v1_threshold_sensitivity.csv + borderline_orb.csv. DONE — send output.")

=== SENSITIVITY: PlantWild v1 ===
                      dup_groups>1  test_leak  test_leak_%  conflict_imgs  clean_test
setting                                                                              
strict only (no ORB)           751        263         7.15            944        3484
strict + ORB>=16              1126        414        11.26           1130        3438
strict + ORB>=50              1101        406        11.04           1126        3442
strict + ORB>=100             1082        396        10.77           1121        3443
strict + ORB>=150             1066        392        10.66           1109        3447
strict + ORB>=286             1015        366         9.95           1091        3452

Inlier distribution of borderline pairs:
inliers
(-1, 10]          967
(10, 16]           24
(16, 50]           31
(50, 100]          35
(100, 150]         41
(150, 286]         99
(286, 1000000]    626

Saved plantwild_v1_threshold_sensitivity.csv + borderline_orb.csv. DONE —

In [10]:
# =====================================================================
# STEP 4.7 (FIXED) — PlantWild v1 <-> v2 content matching
#                    (expert keep / relabel / remove)
# Same session. GPU. ~3-6 min.
# =====================================================================
import zipfile, time
import numpy as np, pandas as pd, torch, timm, imagehash
from pathlib import Path
from PIL import Image, ImageOps

W = Path("/kaggle/working"); RAW = Path("/tmp/plantwild_raw")
V2 = RAW / "v2"
if not V2.exists():
    with zipfile.ZipFile(RAW / "plantwild_v2.zip") as z: z.extractall(V2)
files = sorted(p for p in V2.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
v2 = pd.DataFrame({"path": [str(p) for p in files]})
v2["class_name"] = [p.parent.name for p in files]
v2["fname"] = [p.name for p in files]
print("v2 images:", len(v2), "| classes:", v2.class_name.nunique())

# ---------------- embed + pHash v2 (same preprocessing as v1) ----------------
dev = "cuda"
model = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True,
                          num_classes=0, img_size=224).to(dev).eval().half()
cfg = timm.data.resolve_data_config({}, model=model)
tf = timm.data.create_transform(**{**cfg, "input_size": (3, 224, 224), "crop_pct": 1.0})

class DS(torch.utils.data.Dataset):
    def __len__(self): return len(v2)
    def __getitem__(self, i):
        with Image.open(v2.path[i]) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size; s = min(w, h)
            if s > 320:
                k = 320 / s; im = im.resize((round(w*k), round(h*k)), Image.BICUBIC)
            ph = str(imagehash.phash(im, hash_size=8))      # hex string (collates as list)
            return tf(im), ph

t = time.time()
dl = torch.utils.data.DataLoader(DS(), batch_size=256, num_workers=4)
F, Hs = [], []
with torch.no_grad():
    for x, h in dl:
        F.append(model(x.to(dev).half()).float())
        Hs.append(np.array([int(s, 16) for s in h], dtype=np.uint64))
E2 = torch.nn.functional.normalize(torch.cat(F), dim=1)
H2 = np.concatenate(Hs)
np.save(W / "plantwild_v2_dinov2b_224.npy", E2.cpu().numpy())
np.save(W / "plantwild_v2_phash.npy", H2)
print(f"v2 embedded in {time.time()-t:.0f}s")

# ---------------- nearest v2 match for every v1 image ----------------
df = pd.read_csv(W / "plantwild_v1_manifest.csv"); df = df[df.ok == 1].reset_index(drop=True)
E1 = torch.tensor(np.load(W / "plantwild_v1_dinov2b_224.npy"), device=dev).half()
H1 = np.load(W / "plantwild_v1_phash.npy")
E2h = E2.half()
best_cos, best_j = [], []
for s in range(0, len(E1), 2048):
    S = (E1[s:s+2048] @ E2h.T).float()
    v, j = S.max(1); best_cos.append(v.cpu().numpy()); best_j.append(j.cpu().numpy())
df["v2_cos"] = np.concatenate(best_cos); j = np.concatenate(best_j)
df["v2_ham"] = np.bitwise_count(H1 ^ H2[j]).astype(int)
df["v2_class"] = v2.class_name.values[j]; df["v2_fname"] = v2.fname.values[j]
df["in_v2"] = ((df.v2_ham <= 8) | (df.v2_cos >= 0.98)).astype(int)

# ---------------- filename hypothesis check ----------------
df["v2_name_guess"] = df.class_name.str.replace(" ", "_") + "_" + df.fname
name_hit = df.v2_name_guess.isin(set(v2.fname))
print(f"\nFilename hypothesis (class_underscored + '_' + v1 name) matches: {int(name_hit.sum())} / {len(df)}")
if name_hit.any():
    print(f"  of those, content-confirmed: {(df[name_hit].in_v2 == 1).mean():.3f}")

# ---------------- expert decisions on shared disease classes ----------------
shared = sorted(set(df.class_name) & set(v2.class_name))
d = df[df.class_name.isin(shared)].copy()
d["expert"] = np.where(d.in_v2 == 0, "removed",
               np.where(d.v2_class == d.class_name, "kept", "relabelled"))
print(f"\n=== EXPERT DECISIONS on {len(shared)} shared disease classes ({len(d)} v1 images) ===")
print(d.expert.value_counts().to_string())
print((100 * d.expert.value_counts(normalize=True)).round(1).to_string())

rel = d[d.expert == "relabelled"]
print("\nTop relabelling moves (v1 -> v2):")
print((rel.class_name + "  ->  " + rel.v2_class).value_counts().head(15).to_string())

matched_v2 = set(df[df.in_v2 == 1].v2_fname)
print(f"\nv2 images matched to some v1 image: {len(matched_v2)} / {len(v2)} "
      f"-> new images added in v2: ~{len(v2) - len(matched_v2)}")

# ---------------- does our audit agree with the experts? ----------------
dr = d.drop_reason.fillna("")
d["audit"] = np.select([dr == "label_conflict",
                        dr.str.startswith("dup") | (dr == "leak_copy")],
                       ["label_conflict", "duplicate"], "clean")
tab = pd.crosstab(d.audit, d.expert, normalize="index").mul(100).round(1)
tab["n"] = d.audit.value_counts()
print("\n=== OUR AUDIT vs EXPERT DECISION (row %) ===")
print(tab.to_string())

print("\nPer-class expert non-kept rate (highest 10):")
print((100 * d.groupby("class_name").expert.apply(lambda s: (s != "kept").mean()))
      .sort_values(ascending=False).head(10).round(1).to_string())

df.to_csv(W / "plantwild_v1_manifest.csv", index=False)
v2.to_csv(W / "plantwild_v2_index.csv", index=False)
print("\nManifest updated (in_v2, v2_class, v2_cos, v2_ham). DONE — send ALL output.")

v2 images: 11488 | classes: 115
v2 embedded in 112s

Filename hypothesis (class_underscored + '_' + v1 name) matches: 6279 / 18542
  of those, content-confirmed: 1.000

=== EXPERT DECISIONS on 58 shared disease classes (10585 v1 images) ===
expert
kept          6298
removed       3938
relabelled     349
expert
kept          59.5
removed       37.2
relabelled     3.3

Top relabelling moves (v1 -> v2):
cauliflower alternaria leaf spot  ->  cabbage alternaria leaf spot    14
squash powdery mildew  ->  cucumber powdery mildew                    14
rice sheath blight  ->  ginger sheath blight                          13
corn gray leaf spot  ->  corn northern leaf blight                    11
tomato early blight  ->  potato early blight                          11
tomato bacterial leaf spot  ->  tomato septoria leaf spot              7
tomato bacterial leaf spot  ->  tomato early blight                    7
tomato mosaic virus  ->  tomato yellow leaf curl virus                 7
tomato yello

In [11]:
# =====================================================================
# STEP 4.8 — PlantDoc <-> PlantWild v1: shared-class mapping +
#            cross-dataset duplicate audit
# Same session (PlantWild files in /kaggle/working, PlantDoc dataset attached).
# GPU not required (uses saved embeddings). ~3-5 min.
# =====================================================================
import numpy as np, pandas as pd, imagehash, cv2
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

W = Path("/kaggle/working")
PDROOT = list(Path("/kaggle/input").rglob("plantdoc_manifest.csv"))[0].parent
print("PlantDoc root:", PDROOT)

pdm = pd.read_csv(PDROOT / "plantdoc_manifest.csv")
pdm["proc_path"] = pdm.proc_path.str.replace("/kaggle/working/", str(PDROOT) + "/", regex=False)
assert Path(pdm.proc_path[0]).exists(), "PlantDoc image paths not found"
pd_uids = pd.read_csv(PDROOT / "plantdoc_dinov2b_224_uids.csv").uid
E_pd = np.load(PDROOT / "plantdoc_dinov2b_224.npy")
pdm = pdm.set_index("uid").loc[pd_uids].reset_index()          # align with embeddings

pwm = pd.read_csv(W / "plantwild_v1_manifest.csv"); pwm = pwm[pwm.ok == 1].reset_index(drop=True)
E_pw = np.load(W / "plantwild_v1_dinov2b_224.npy")
H_pw = np.load(W / "plantwild_v1_phash.npy")
pw_classes = sorted(pwm.class_name.unique())

# ---------------- 1) proposed mapping ----------------
MAP = {
 "Apple Scab Leaf": "apple scab", "Apple leaf": "apple leaf", "Apple rust leaf": "apple rust",
 "Bell_pepper leaf": "bell pepper leaf", "Bell_pepper leaf spot": "bell pepper leaf spot",
 "Blueberry leaf": "blueberry leaf", "Cherry leaf": "cherry leaf",
 "Corn Gray leaf spot": "corn gray leaf spot", "Corn leaf blight": "corn northern leaf blight",
 "Corn rust leaf": "corn rust", "Peach leaf": "peach leaf",
 "Potato leaf early blight": "potato early blight", "Potato leaf late blight": "potato late blight",
 "Raspberry leaf": "raspberry leaf", "Soyabean leaf": "soybean leaf",
 "Squash Powdery mildew leaf": "squash powdery mildew", "Strawberry leaf": "strawberry leaf",
 "Tomato Early blight leaf": "tomato early blight", "Tomato Septoria leaf spot": "tomato septoria leaf spot",
 "Tomato leaf": "tomato leaf", "Tomato leaf bacterial spot": "tomato bacterial leaf spot",
 "Tomato leaf late blight": "tomato late blight", "Tomato leaf mosaic virus": "tomato mosaic virus",
 "Tomato leaf yellow virus": "tomato yellow leaf curl virus", "Tomato mold leaf": "tomato leaf mold",
 "grape leaf": "grape leaf", "grape leaf black rot": "grape black rot",
}
missing = {k: v for k, v in MAP.items() if v not in pw_classes}
print("\nMapped PlantDoc classes:", len(MAP), "| targets missing in PlantWild:", missing)
if missing:
    for k, v in missing.items():
        cand = [c for c in pw_classes if any(w in c for w in v.split()[:2])]
        print(f"  {k!r} -> {v!r} not found. Similar names: {cand[:8]}")

# ---------------- 2) centroid sanity check ----------------
def centroids(E, labels):
    out = {}
    for c in sorted(set(labels)):
        m = E[labels == c].mean(0); out[c] = m / np.linalg.norm(m)
    return out
C_pd = centroids(E_pd, pdm.class_name.values)
C_pw = centroids(E_pw, pwm.class_name.values)
pw_names = list(C_pw); Mpw = np.stack([C_pw[c] for c in pw_names])
rows = []
for pdc, v in C_pd.items():
    sims = Mpw @ v; top = np.argsort(-sims)[:3]
    tgt = MAP.get(pdc)
    rows.append({"plantdoc": pdc, "mapped_to": tgt,
                 "sim_mapped": round(float(sims[pw_names.index(tgt)]), 3) if tgt in pw_names else None,
                 "rank_of_mapped": int((sims > sims[pw_names.index(tgt)]).sum()) + 1 if tgt in pw_names else None,
                 "top1": f"{pw_names[top[0]]} ({sims[top[0]]:.3f})",
                 "top2": f"{pw_names[top[1]]} ({sims[top[1]]:.3f})",
                 "top3": f"{pw_names[top[2]]} ({sims[top[2]]:.3f})",
                 "n_pd_clean": int(((pdm.class_name == pdc) & (pdm.in_clean == 1)).sum()),
                 "n_pw_clean": int(((pwm.class_name == tgt) & (pwm.in_clean == 1)).sum())})
mt = pd.DataFrame(rows).sort_values("rank_of_mapped", ascending=False)
pd.set_option("display.width", 250)
print("\n=== MAPPING SANITY (rank_of_mapped = 1 is ideal) ===")
print(mt.to_string(index=False))
print("\nMapped class is nearest PlantWild centroid for",
      int((mt.rank_of_mapped == 1).sum()), "/", len(mt), "PlantDoc classes")
mt.to_csv(W / "pd_pw_class_mapping.csv", index=False)

# ---------------- 3) cross-dataset duplicates ----------------
def ph(p):
    with Image.open(p) as im:
        return int(str(imagehash.phash(im.convert("RGB"), hash_size=8)), 16)
with ThreadPoolExecutor(max_workers=8) as ex:
    H_pd = np.array(list(ex.map(ph, pdm.proc_path)), dtype=np.uint64)

S = E_pd @ E_pw.T                                   # 2576 x 18542
Hd = np.bitwise_count(H_pd[:, None] ^ H_pw[None, :])
ii, jj = np.nonzero((S >= 0.93) | (Hd <= 8))
X = pd.DataFrame({"i": ii, "j": jj, "cos": S[ii, jj], "ham": Hd[ii, jj]})
print(f"\nCross-dataset candidate pairs: {len(X)}")

orb = None
def inl(a, b, flip):
    g1 = cv2.imread(pdm.proc_path[a], 0); g2 = cv2.imread(pwm.proc_path[b], 0)
    if flip: g2 = cv2.flip(g2, 1)
    o = cv2.ORB_create(nfeatures=1500)
    k1, d1 = o.detectAndCompute(g1, None); k2, d2 = o.detectAndCompute(g2, None)
    if d1 is None or d2 is None or len(k1) < 8 or len(k2) < 8: return 0
    m = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(d1, d2, k=2)
    good = [x for x, *y in m if y and x.distance < 0.75 * y[0].distance]
    if len(good) < 8: return len(good) // 2
    s = np.float32([k1[g.queryIdx].pt for g in good]).reshape(-1, 1, 2)
    d = np.float32([k2[g.trainIdx].pt for g in good]).reshape(-1, 1, 2)
    _, mk = cv2.findHomography(s, d, cv2.RANSAC, 5.0)
    return 0 if mk is None else int(mk.sum())
bandX = X[(X.ham > 8) & (X.cos < 0.98)]
with ThreadPoolExecutor(max_workers=4) as ex:
    inl_b = list(ex.map(lambda t: max(inl(*t, False), inl(*t, True)),
                        bandX[["i", "j"]].values.tolist()))
X["inliers"] = np.nan; X.loc[bandX.index, "inliers"] = inl_b
X["dup"] = (X.ham <= 8) | (X.cos >= 0.98) | (X.inliers >= 16)
D = X[X.dup].copy()
D["pd_class"] = pdm.class_name.values[D.i]; D["pw_class"] = pwm.class_name.values[D.j]
D["pd_role_p1"] = pdm.p1_role.values[D.i]; D["pw_role"] = pwm.official_role.values[D.j]
D["label_agree"] = D.pd_class.map(MAP) == D.pw_class

print(f"\n=== CROSS-DATASET DUPLICATES ===")
print(f"pairs: {len(D)} | unique PlantDoc images: {D.i.nunique()} / {len(pdm)} "
      f"({100*D.i.nunique()/len(pdm):.1f}%) | unique PlantWild images: {D.j.nunique()}")
print("Label agreement (after mapping):", D.label_agree.value_counts().to_dict())
for name, mask in [("PlantDoc official TEST (236)", pdm.p1_role == "test"),
                   ("PlantDoc clean TEST (220)", pdm.p2_role == "test")]:
    idx = set(np.where(mask)[0]); hit = idx & set(D.i)
    print(f"{name}: {len(hit)} have a twin in PlantWild ({100*len(hit)/len(idx):.1f}%)")
pw_test = set(np.where(pwm.official_role == "test")[0]); hit = pw_test & set(D.j)
print(f"PlantWild official TEST (3677): {len(hit)} have a twin in PlantDoc ({100*len(hit)/len(pw_test):.1f}%)")
print("\nTop disagreeing label pairs (PlantDoc -> PlantWild):")
print((D[~D.label_agree].pd_class + "  ->  " + D[~D.label_agree].pw_class).value_counts().head(12).to_string())
print("\nDuplicates by mapped PlantDoc class (top 10):")
print(D.drop_duplicates("i").pd_class.value_counts().head(10).to_string())

D["pd_uid"] = pdm.uid.values[D.i]; D["pw_uid"] = pwm.uid.values[D.j]
D.to_csv(W / "pd_pw_cross_duplicates.csv", index=False)
np.save(W / "plantdoc_phash.npy", H_pd)
print("\nSaved pd_pw_class_mapping.csv, pd_pw_cross_duplicates.csv. DONE — send ALL output.")

PlantDoc root: /kaggle/input/datasets/mashfii/plantdoc-clean-thesis

Mapped PlantDoc classes: 27 | targets missing in PlantWild: {}

=== MAPPING SANITY (rank_of_mapped = 1 is ideal) ===
                  plantdoc                     mapped_to  sim_mapped  rank_of_mapped                                  top1                                  top2                                  top3  n_pd_clean  n_pw_clean
      grape leaf black rot               grape black rot       0.711               7               grape leaf spot (0.963)            grape downy mildew (0.948)    grapevine leafroll disease (0.912)          60         207
   Tomato leaf late blight            tomato late blight       0.874               5              tomato leaf mold (0.958)           tomato early blight (0.949)     tomato septoria leaf spot (0.935)          99         251
           Apple Scab Leaf                    apple scab       0.808               3              cherry leaf spot (0.885)            apple mosai

In [12]:
# =====================================================================
# STEP 4.9 — Final protocol files + thesis-ready audit tables
#   (a) PlantDoc ORB-threshold sensitivity (symmetry with PlantWild)
#   (b) PlantWild expert-verified test subset flag
#   (c) Cross-dataset protocol (27 shared classes, twins removed)
#   (d) Audit summary table + README
# Add as the LAST cell of the PlantWild notebook (after 4.8), then commit.
# Reads only saved files -> safe for "Save & Run All". CPU, ~2-4 min.
# =====================================================================
import numpy as np, pandas as pd, cv2, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

W = Path("/kaggle/working")
PDROOT = list(Path("/kaggle/input").rglob("plantdoc_manifest.csv"))[0].parent
pdm = pd.read_csv(PDROOT / "plantdoc_manifest.csv")
pdm["proc_path"] = pdm.proc_path.str.replace("/kaggle/working/", str(PDROOT) + "/", regex=False)
pwm = pd.read_csv(W / "plantwild_v1_manifest.csv"); pwm = pwm[pwm.ok == 1].reset_index(drop=True)
MAPT = pd.read_csv(W / "pd_pw_class_mapping.csv")
MAP = dict(zip(MAPT.plantdoc, MAPT.mapped_to))
X = pd.read_csv(W / "pd_pw_cross_duplicates.csv")

# ---------------- (a) PlantDoc sensitivity ----------------
cand = pd.read_csv(PDROOT / "plantdoc_dup_candidates.csv")
u2i = {u: k for k, u in enumerate(pdm.uid)}
cand["i"] = cand.uid_i.map(u2i); cand["j"] = cand.uid_j.map(u2i)
def inl(a, b, flip):
    g1 = cv2.imread(pdm.proc_path[a], 0); g2 = cv2.imread(pdm.proc_path[b], 0)
    if flip: g2 = cv2.flip(g2, 1)
    o = cv2.ORB_create(nfeatures=1500)
    k1, d1 = o.detectAndCompute(g1, None); k2, d2 = o.detectAndCompute(g2, None)
    if d1 is None or d2 is None or len(k1) < 8 or len(k2) < 8: return 0
    m = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(d1, d2, k=2)
    good = [x for x, *y in m if y and x.distance < 0.75 * y[0].distance]
    if len(good) < 8: return len(good) // 2
    s = np.float32([k1[g.queryIdx].pt for g in good]).reshape(-1, 1, 2)
    d = np.float32([k2[g.trainIdx].pt for g in good]).reshape(-1, 1, 2)
    _, mk = cv2.findHomography(s, d, cv2.RANSAC, 5.0)
    return 0 if mk is None else int(mk.sum())
strict = cand[(cand.ham <= 8) | (cand.cos >= 0.98)]
band = cand[(cand.cos >= 0.93) & (cand.cos < 0.98) & (cand.ham > 8)].copy()
with ThreadPoolExecutor(max_workers=4) as ex:
    band["inliers"] = list(ex.map(lambda t: max(inl(*t, False), inl(*t, True)),
                                  band[["i", "j"]].values.tolist()))

def evaluate(edges, base, role_col):
    N = len(base); parent = list(range(N))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    t = base.assign(g=[find(k) for k in range(N)])
    ncls = t.groupby("g").class_name.nunique(); nrole = t.groupby("g")[role_col].nunique()
    conflict = t.g.isin(ncls[ncls > 1].index)
    leak = ((t[role_col] == "test") & t.g.isin(nrole[nrole > 1].index)).sum()
    ntest = (t[role_col] == "test").sum()
    return {"test_leak": int(leak), "test_leak_%": round(100 * leak / ntest, 2),
            "conflict_imgs": int(conflict.sum()),
            "clean_test": int(t[(t[role_col] == "test") & ~conflict].g.nunique())}
rows = []
se = list(map(tuple, strict[["i", "j"]].values))
rows.append({"setting": "strict only (no ORB)", **evaluate(se, pdm, "orig_split")})
for thr in [16, 50, 100, 150, 286]:
    e = se + list(map(tuple, band[band.inliers >= thr][["i", "j"]].values))
    rows.append({"setting": f"strict + ORB>={thr}", **evaluate(e, pdm, "orig_split")})
sens_pd = pd.DataFrame(rows).set_index("setting")
print("=== (a) SENSITIVITY: PlantDoc ==="); print(sens_pd.to_string())
sens_pd.to_csv(W / "plantdoc_threshold_sensitivity.csv")

# ---------------- (b) PlantWild expert-verified subset ----------------
v2cls = set(pd.read_csv(W / "plantwild_v2_index.csv").class_name)
shared = pwm.class_name.isin(v2cls)
pwm["expert_status"] = np.where(~shared, "not_in_v2_classes",
                        np.where(pwm.in_v2 == 0, "removed",
                        np.where(pwm.v2_class == pwm.class_name, "kept", "relabelled")))
pwm["expert_verified_test"] = ((pwm.official_role == "test") & (pwm.expert_status == "kept")).astype(int)
ev = pwm[pwm.expert_verified_test == 1]
print("\n=== (b) EXPERT-VERIFIED TEST SUBSET ===")
print(f"images: {len(ev)} | classes: {ev.class_name.nunique()} (disease classes shared with v2) | "
      f"min per class: {ev.class_name.value_counts().min()} | also in clean test: {int((ev.in_clean == 1).sum())}")

# ---------------- (c) cross-dataset protocol ----------------
ORGAN_SHIFT = {"grape leaf black rot", "Apple Scab Leaf", "Apple rust leaf", "Tomato leaf late blight"}
MAPT["organ_shift"] = MAPT.plantdoc.isin(ORGAN_SHIFT).astype(int)
MAPT.to_csv(W / "pd_pw_class_mapping.csv", index=False)
inv = {v: k for k, v in MAP.items()}
pd_twin = set(X.pd_uid); pw_twin = set(X.pw_uid)
pw_test_uids = set(pwm.uid[pwm.official_role == "test"])
pd_twin_of_pwtest = set(X.pd_uid[X.pw_uid.isin(pw_test_uids)])

# PW -> PD : train on PlantWild (27 mapped classes, clean, no twin in PlantDoc) ; test on PlantDoc
pwm["xd_class"] = pwm.class_name.map(inv)            # PlantDoc-style label (NaN if unmapped)
pwm["xd_pw2pd_role"] = np.where(pwm.xd_class.notna() & (pwm.in_clean == 1) &
                                ~pwm.uid.isin(pw_twin) & pwm.p2_role.isin(["train", "val"]),
                                pwm.p2_role, "excluded")
# PD -> PW : train on PlantDoc clean (no twin in PlantWild test) ; test on PlantWild clean test (mapped, no twin in PlantDoc)
pdm["xd_pd2pw_role"] = np.where(pdm.p2_role.isin(["train", "val"]) & ~pdm.uid.isin(pd_twin_of_pwtest),
                                pdm.p2_role, "excluded")
pwm["xd_pd2pw_test"] = ((pwm.p2_role == "test") & pwm.xd_class.notna() &
                        ~pwm.uid.isin(pw_twin)).astype(int)
pdm["xd_pw2pd_test_clean"] = ((pdm.p2_role == "test") & ~pdm.uid.isin(pd_twin)).astype(int)
print("\n=== (c) CROSS-DATASET PROTOCOL (27 shared classes) ===")
print("PW->PD  train/val on PlantWild:", pwm.xd_pw2pd_role.value_counts().to_dict())
print(f"        test on PlantDoc clean test: {int((pdm.p2_role=='test').sum())} "
      f"(twin-free subset: {int(pdm.xd_pw2pd_test_clean.sum())})")
print("PD->PW  train/val on PlantDoc :", pdm.xd_pd2pw_role.value_counts().to_dict())
print(f"        test on PlantWild clean mapped test (twin-free): {int(pwm.xd_pd2pw_test.sum())}")
print("Organ-shift classes flagged:", sorted(ORGAN_SHIFT))

# ---------------- (d) audit summary ----------------
pw_sens = pd.read_csv(W / "plantwild_v1_threshold_sensitivity.csv", index_col=0)
summary = pd.DataFrame([
 {"dataset": "PlantDoc", "images": len(pdm), "classes": pdm.class_name.nunique(),
  "official_test": int((pdm.orig_split == "test").sum()),
  "test_with_twin_in_train_%": sens_pd.loc["strict + ORB>=286", "test_leak_%"],
  "test_leak_range_%": f"{sens_pd['test_leak_%'].min()}-{sens_pd['test_leak_%'].max()}",
  "label_conflict_imgs": int((pdm.drop_reason == "label_conflict").sum()),
  "clean_train": int(((pdm.orig_split == "train") & (pdm.in_clean == 1)).sum()),
  "clean_test": int(((pdm.orig_split == "test") & (pdm.in_clean == 1)).sum()),
  "twins_in_other_dataset_%": round(100 * len(pd_twin) / len(pdm), 1)},
 {"dataset": "PlantWild v1", "images": len(pwm), "classes": pwm.class_name.nunique(),
  "official_test": int((pwm.official_role == "test").sum()),
  "test_with_twin_in_train_%": pw_sens.loc["strict + ORB>=16", "test_leak_%"],
  "test_leak_range_%": f"{pw_sens['test_leak_%'].min()}-{pw_sens['test_leak_%'].max()}",
  "label_conflict_imgs": int((pwm.drop_reason == "label_conflict").sum()),
  "clean_train": int(((pwm.official_role == "train") & (pwm.in_clean == 1)).sum()),
  "clean_test": int(((pwm.official_role == "test") & (pwm.in_clean == 1)).sum()),
  "twins_in_other_dataset_%": round(100 * len(pw_twin) / len(pwm), 1)},
]).set_index("dataset")
print("\n=== (d) AUDIT SUMMARY (thesis Table) ===")
print(summary.T.to_string())
summary.to_csv(W / "audit_summary_table.csv")

pwm.to_csv(W / "plantwild_v1_manifest.csv", index=False)
pdm.to_csv(W / "plantdoc_manifest_xd.csv", index=False)   # PlantDoc + cross-dataset columns
(W / "README_plantwild.txt").write_text(
"""PlantWild v1 processed (PRIVATE - source licence CC BY-NC-ND 4.0, do not redistribute images)
Source: huggingface.co/datasets/uqtwei2/PlantWild (plantwild.zip v1, 18,542 imgs, 89 classes)
Official split: trainval.txt modes 0=test 1=train 2=val (official_role / p1_role)
Processing: EXIF transpose, RGB, short side 320 (no upscaling), JPEG q95
Duplicates: pHash<=8 OR DINOv2-B cos>=0.98 OR (0.93<=cos<0.98 AND ORB-RANSAC inliers>=16)
Clean (p2_role): label-conflict groups removed; 1 image kept per duplicate group (priority test>val>train)
Expert columns: in_v2, v2_class, expert_status, expert_verified_test (matched against plantwild_v2.zip)
Cross-dataset: xd_class, xd_pw2pd_role, xd_pd2pw_test (+ plantdoc_manifest_xd.csv, pd_pw_class_mapping.csv)
NOTE: 'Copy'-named files are filename collisions, NOT duplicates (verified).
""")
print("\nSaved: manifests, sensitivity tables, audit_summary_table.csv, README. DONE.")

=== (a) SENSITIVITY: PlantDoc ===
                      test_leak  test_leak_%  conflict_imgs  clean_test
setting                                                                
strict only (no ORB)         17         7.20             71         226
strict + ORB>=16             32        13.56            102         219
strict + ORB>=50             31        13.14            100         220
strict + ORB>=100            30        12.71            100         220
strict + ORB>=150            30        12.71            100         220
strict + ORB>=286            27        11.44             94         220

=== (b) EXPERT-VERIFIED TEST SUBSET ===
images: 1232 | classes: 58 (disease classes shared with v2) | min per class: 3 | also in clean test: 1151

=== (c) CROSS-DATASET PROTOCOL (27 shared classes) ===
PW->PD  train/val on PlantWild: {'excluded': 14288, 'train': 3727, 'val': 527}
        test on PlantDoc clean test: 220 (twin-free subset: 173)
PD->PW  train/val on PlantDoc : {'train': 1

In [13]:
# STEP 4.10 — pack processed images into ONE zip so dataset creation works
import shutil, os
from pathlib import Path
W = Path("/kaggle/working")
src = W / "proc" / "plantwild_v1"
shutil.make_archive(str(W / "plantwild_v1_proc"), "zip", root_dir=W / "proc", base_dir="plantwild_v1")
shutil.rmtree(W / "proc", ignore_errors=True)
files = [p for p in W.rglob("*") if p.is_file()]
print(f"Files in output: {len(files)} | size: {sum(p.stat().st_size for p in files)/1e9:.2f} GB")
for p in sorted(W.iterdir()): print("  ", p.name)

Files in output: 22 | size: 1.19 GB
   .virtual_documents
   README_plantwild.txt
   audit_summary_table.csv
   pd_pw_class_mapping.csv
   pd_pw_cross_duplicates.csv
   plantdoc_manifest_xd.csv
   plantdoc_phash.npy
   plantdoc_threshold_sensitivity.csv
   plantwild_v1_borderline_orb.csv
   plantwild_v1_classes.json
   plantwild_v1_dinov2b_224.npy
   plantwild_v1_dinov2b_224_uids.csv
   plantwild_v1_dup_candidates.csv
   plantwild_v1_dup_edges_final.csv
   plantwild_v1_manifest.csv
   plantwild_v1_phash.npy
   plantwild_v1_proc.zip
   plantwild_v1_split_raw.csv
   plantwild_v1_threshold_sensitivity.csv
   plantwild_v2_dinov2b_224.npy
   plantwild_v2_index.csv
   plantwild_v2_phash.npy
